<a href="https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd
import numpy as np

fatal: destination path 'FlyRank-ML' already exists and is not an empty directory.


In [2]:
df = pd.read_csv('/content/FlyRank-ML/data/raw/content_refresh_anonymized.csv')

## 1. My rule and its reason codes

A page is worth reviewing first if it used to bring in
real traffic, and it hasn't been touched in a long time. "Used to bring in traffic"
is measured by impressions_prev_30d — the 30 days before the most recent window,
never the window closest to now, to stay clear of the leakage boundary from ML-05.
"Hasn't been touched" is measured by freshness_tier — pages that haven't been
updated in 91+ days.

Both conditions are gates, not weights: a page has to clear both, and once it does,
pages are ranked by how much traffic it had going for it — the bigger the audience
at stake, the higher the priority.

**Reason codes:**
- `stale_and_visible` — cleared both gates, ranked by impressions_prev_30d
- `not_stale` — recently updated (freshness_tier 0-30 or 31-90), not flagged
  regardless of traffic
- `not_visible_enough` — stale, but impressions_prev_30d < 500, not enough
  audience at stake to prioritize

In [4]:
stale = df['freshness_tier'].isin(['91-180', '181+']).astype(int)
visible = (df['impressions_prev_30d'] >= 500).astype(int)
df['score'] = stale * visible * df['impressions_prev_30d']

def reason(row_stale, row_visible):
    if not row_stale:
        return 'not_stale'
    elif not row_visible:
        return 'not_visible_enough'
    else:
        return 'stale_and_visible'

df['reason_code'] = [reason(s, v) for s, v in zip(stale, visible)]

# --- Precision@K, always next to the base rate ---
df['label_down'] = (df['trend_direction'] == 'down').astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df['label_down'].mean()
print(f"base rate (share of ALL pages that are 'down'): {base_rate:.3f}")
for k in [20, 50, 100, 500, 1000]:
    print(f"precision@{k}: {precision_at_k(df['score'], df['label_down'], k):.3f}")

# --- dummy floor: majority class ---
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(df[['impressions_prev_30d']], df['label_down'])
print("dummy (always predict majority):", dummy.score(df[['impressions_prev_30d']], df['label_down']))

# --- write the ranked queue ---
ranked = df.sort_values('score', ascending=False)[
    ['content_id', 'score', 'reason_code', 'freshness_tier',
     'days_since_last_update', 'impressions_prev_30d', 'content_age_days',
     'age_tier', 'trend_direction']
]

base rate (share of ALL pages that are 'down'): 0.542
precision@20: 0.600
precision@50: 0.460
precision@100: 0.440
precision@500: 0.490
precision@1000: 0.539
dummy (always predict majority): 0.5420666666666667


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.